# Prepare meteo data 2023 for submission

The 2m Lufft instrument was destroyed and not available between mid-October 2023 and April 2024. 
The Rotronic data are available for the entire period, but are not aggregated on the DWH (why not?). 
We will use the official hourly data from DWH where available and complement with our own aggregated Rotronic data where needed.

author: joerg.klausen@meteoswiss.ch

In [6]:
import os
import polars as pl
from processing.dwh import DWH

access_token = open(file="secrets-jretrieve", mode="r").read()
mkn = DWH(access_token=access_token, locationID="KEMKN")

DWH initialized.


In [7]:
metadata = {
    'prestah0': '2m pressure (QFE), Lufft (hPa)',
    'ta2200h0': '2m temperature, Rotronic (°C)',
    'ua2200h0': '2m relative humidity, Rotronic (%)',
    'fkl010h0': '10m horizontal wind speed (m/s)',
    'dkl010h0': '10m horizontal wind direction (deg)',
    'rre150h0': '2m precipitation, Lufft (mm/h)',
    'gre000h0': '2m global radiation, Lufft (W/m2)',
    # 'tre200h0': '2m temperature, Lufft (up until 19 Oct 2023 09 UTC)',
    # 'ure200h0': '2m relative humidity, Lufft (%) (up until 19 Oct 2023 09 UTC)',
}

In [12]:
# get hourly values from MeteoSwiss DWH
parameter_short_names = "prestah0,tre200h0,ure200h0,fkl010h0,dkl010h0,rre150h0,gre000h0"
df1_1h = mkn.jretrieve(start="20230101000000", end="20240101000000", parameter_short_names=parameter_short_names)

# get Rotronic data and temperature, RH data from 10m sensor from MeteoSwiss DWH and aggregate
parameter_short_names = "ta2200s0,ua2200s0,ta1200s0,ua1200s0"
df2 = mkn.jretrieve(start="20230101000000", end="20240101000000", parameter_short_names=parameter_short_names)

# df2_1h = df2.sort(by='dtm').group_by_dynamic("dtm", every='1h').agg(pl.all().exclude(['dtm', 'termin', 'station']).mean())
# df2_1h = df2.sort(by='dtm').group_by_dynamic("dtm", every='1h', closed='right').agg(pl.all().exclude(['dtm', 'termin', 'station']).mean())
df2_1h = df2.sort(by='dtm').group_by_dynamic("dtm", every='1h', closed='right', label='right').agg(pl.all().exclude(['dtm', 'termin', 'station']).mean())
df2_1h = df2_1h.rename({'ta2200s0': 'ta2200h0', 'ua2200s0': 'ua2200h0', 'ta1200s0': 'ta1200h0', 'ua1200s0': 'ua1200h0'})

In [13]:
# combine dataframes, compute biases
df_1h = pl.concat([df1_1h, df2_1h], how='align')
df_1h.drop_in_place('station')
df_1h.drop_in_place('termin')
df_1h = df_1h.with_columns((pl.col('tre200h0')-pl.col('ta2200h0')).alias('tre-ta2'),
                   (pl.col('ta1200h0')-pl.col('ta2200h0')).alias('ta1-ta2'),
                   (pl.col('ure200h0')-pl.col('ua2200h0')).alias('ure-ua2'),
                   (pl.col('ua1200h0')-pl.col('ua2200h0')).alias('ua1-ua2'),
)
display(df_1h.describe())

statistic,prestah0,tre200h0,ure200h0,fkl010h0,dkl010h0,rre150h0,gre000h0,dtm,ta2200h0,ua2200h0,ta1200h0,ua1200h0,tre-ta2,ta1-ta2,ure-ua2,ua1-ua2
str,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",5659.0,5653.0,5640.0,7292.0,7235.0,5619.0,5643.0,"""7321""",7310.0,7281.0,7310.0,7278.0,5653.0,7310.0,5628.0,7278.0
"""null_count""",1662.0,1668.0,1681.0,29.0,86.0,1702.0,1678.0,"""0""",11.0,40.0,11.0,43.0,1668.0,11.0,1693.0,43.0
"""mean""",661.703958,6.755563,70.163404,4.484202,164.729371,0.133992,195.220273,"""2023-07-12 04:…",6.897404,76.773748,7.019993,74.882374,-0.017668,0.12259,-3.498534,-1.883815
"""std""",1.133786,2.492184,18.45749,2.601227,94.379702,1.449551,280.142282,null,2.458905,20.533272,1.321269,19.047798,0.21267,1.669629,3.01013,6.070307
"""min""",657.8,0.3,10.1,0.0,1.0,0.0,0.0,"""2023-01-01 00:…",0.2,8.966667,2.466667,10.933333,-2.166667,-3.783333,-10.85,-35.1
"""25%""",661.0,4.8,58.6,2.4,110.0,0.0,0.0,"""2023-04-19 02:…",4.95,63.416667,6.183333,62.366667,-0.116667,-1.183333,-5.9,-6.15
"""50%""",661.8,6.4,74.4,3.9,138.0,0.0,16.0,"""2023-07-18 03:…",6.666667,81.533333,6.983333,78.333333,-8.8818e-16,-0.166667,-3.333333,-1.516667
"""75%""",662.5,8.7,84.5,6.2,228.0,0.0,322.0,"""2023-10-11 00:…",8.716667,95.0,7.816667,90.483333,0.1,1.333333,-0.916667,1.85
"""max""",665.2,13.8,99.9,18.8,360.0,43.5,1200.0,"""2024-01-01 00:…",14.0,100.0,12.266667,100.0,1.183333,5.466667,8.75,17.3


In [ ]:
# plot data
import matplotlib.pyplot as plt
%matplotlib widget

fig, axs = plt.subplots(figsize=(10, 20), nrows=9, sharex=True)
axs[0].scatter(x=df_1h['dtm'], y=df_1h['prestah0'], s=5, label='prestah0')
axs[0].legend()
axs[1].scatter(x=df_1h['dtm'], y=df_1h['ta2200h0'], s=5, label='ta2200h0')
axs[1].scatter(x=df_1h['dtm'], y=df_1h['tre200h0'], s=5, c='r', label='tre200h0')
axs[1].scatter(x=df_1h['dtm'], y=df_1h['ta1200h0'], s=5, c='g', label='ta1200h0')
axs[1].legend()
axs[2].scatter(x=df_1h['dtm'], y=df_1h['tre-ta2'], s=5, c='c', label='tre200h0 - ta2200h0')
axs[2].scatter(x=df_1h['dtm'], y=df_1h['ta1-ta2'], s=5, c='m', label='ta1200h0 - ta2200h0')
axs[2].axhline()
axs[2].legend()
axs[3].scatter(x=df_1h['dtm'], y=df_1h['ua2200h0'], s=5, label='ua2200h0')
axs[3].scatter(x=df_1h['dtm'], y=df_1h['ure200h0'], s=5, c='r', label='ure200h0')
axs[3].scatter(x=df_1h['dtm'], y=df_1h['ua1200h0'], s=5, c='g', label='ua1200h0')
axs[3].legend()
axs[4].scatter(x=df_1h['dtm'], y=df_1h['ure-ua2'], s=5, c='c', label='ure200h0 - ua2200h0')
axs[4].scatter(x=df_1h['dtm'], y=df_1h['ua1-ua2'], s=5, c='m', label='ua1200h0 - ua2200h0')
axs[4].axhline()
axs[4].legend()
axs[5].scatter(x=df_1h['dtm'], y=df_1h['fkl010h0'], s=5, label='fkl010h0')
axs[5].legend()
axs[6].scatter(x=df_1h['dtm'], y=df_1h['dkl010h0'], s=5, label='dkl010h0')
axs[6].legend()
axs[7].scatter(x=df_1h['dtm'], y=df_1h['gre000h0'], s=5, label='gre000h0')
axs[7].legend()
axs[8].scatter(x=df_1h['dtm'], y=df_1h['rre150h0'], s=5, label='rre150h0')
axs[8].legend()

In [ ]:
# drop original columns, retain combined columns, save to file
df_1h = df_1h.select([pl.col('dtm'), pl.col('prestah0'), pl.col('ta2200h0'), pl.col('ua2200h0'),
                      pl.col('fkl010h0'), pl.col('dkl010h0'), pl.col('gre000h0'), pl.col('rre150h0'),
                      ])
display(df_1h.describe())

path = "data/level2/2023"
file = "mkn_meteo_1h"
df_1h.write_parquet(os.path.join(path, f"{file}.parquet"))
df_1h.write_csv(os.path.join(path, f"{file}.csv"))

In [ ]:
# add metadata file
import json
with open(file=os.path.join(path, f"{file}.json"), mode='w') as fh:
    fh.write(json.dumps(metadata))

In [ ]:
# load data from .parquet file. These are the data compiled from the original bulletins
# file = "data/level2/2023/vrxa00.parquet"
# df2 = pl.read_parquet(file)
# vrxa00_stats = df2.describe()
# vrxa00_stats